In [18]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [ ]:
class QuadState(TypedDict):
     a: int
     b: int
     c: int
     equation: str
     discriminant: float
     result: str



In [28]:
def show_equation(state: QuadState):
    equation= f"{state['a']}x^2 + {state['b']}x + {state['c']} = 0"
    return {"equation": equation}

def calculate_discriminant(state: QuadState):
    discriminant = state['b']**2 - (4*state['a']*state['c'])
    return {"discriminant": discriminant}

def real_roots(state: QuadState):
    root1 = (-state['b'] + state['discriminant']**0.5) / (2*state['a'])
    root2 = (-state['b'] - state['discriminant']**0.5) / (2*state['a'])
    result = f"The roots are real and distinct: {root1} and {root2}"
    return {"result": result}

def no_real_roots(state: QuadState):
    result = "The equation has no real roots."
    return {"result": result}

def repeated_roots(state: QuadState):
    root = -state['b'] / (2*state['a'])
    result = f"The equation has a repeated root: {root}"
    return {"result": result}

def check_condition(state: QuadState) -> Literal["real_roots", "no_real_roots", "repeated_roots"]:
    if state['discriminant'] > 0:
        return "real_roots"
    elif state['discriminant'] == 0:
        return "repeated_roots"
    else:
        return "no_real_roots"


In [29]:
graph = StateGraph(QuadState)

#add nodes to the graph
graph.add_node('show_equation', show_equation)
graph.add_node('calculate_discriminant', calculate_discriminant)
graph.add_node('real_roots', real_roots)
graph.add_node('no_real_roots', no_real_roots)
graph.add_node('repeated_roots', repeated_roots)


#add edges to the graph
graph.add_edge(START, 'show_equation')
graph.add_edge('show_equation', 'calculate_discriminant')
graph.add_conditional_edges('calculate_discriminant', check_condition)

graph.add_edge('real_roots', END)
graph.add_edge('no_real_roots', END)
graph.add_edge('repeated_roots', END)

workflow = graph.compile()

In [30]:
initial_state = {
    'a': 1,
    'b': -3,
    'c': 2
}

workflow.invoke(initial_state)


{'a': 1,
 'b': -3,
 'c': 2,
 'equation': '1x^2 + -3x + 2 = 0',
 'discriminant': 1,
 'result': 'The roots are real and distinct: 2.0 and 1.0'}